# DMN Colab - Backtest

Notebook Colab pour lancer le backtest global du projet via `python -m dmn.cli`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/figaroldamien/DMN.git"
REPO_DIR = Path("/content/DMN")
DRIVE_DIR = Path("/content/drive/MyDrive/DMN")
CONFIGS_DIR = DRIVE_DIR / "configs"

CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")
print(f"Configs directory: {CONFIGS_DIR}")

In [ ]:
CONFIG_PATH = str(CONFIGS_DIR / "backtest.toml")
USE_CONFIG_FILE = False
MARKET = "cac40"
TICKER = None
START = "2000-01-01"
SECTOR = None
SUB_SECTOR = None
RUN_ML = False
RUN_DMN = True
MODEL_HIDDEN = 64
MODEL_DROPOUT = 0.5
PRINT_CONFIG = True

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_DIR / 'src'}:{env.get('PYTHONPATH', '')}"

cmd = [sys.executable, "-m", "dmn.cli"]
if USE_CONFIG_FILE:
    cmd.extend(["--config", CONFIG_PATH])
if TICKER:
    cmd.extend(["--ticker", TICKER])
else:
    cmd.extend(["--market", MARKET])
if START:
    cmd.extend(["--start", START])
if SECTOR:
    cmd.extend(["--sector", SECTOR])
if SUB_SECTOR:
    cmd.extend(["--sub-sector", SUB_SECTOR])
cmd.extend(["--model-hidden", str(MODEL_HIDDEN), "--model-dropout", str(MODEL_DROPOUT)])
cmd.append("--run-ml" if RUN_ML else "--no-run-ml")
cmd.append("--run-dmn" if RUN_DMN else "--no-run-dmn")
if not PRINT_CONFIG:
    cmd.append("--no-print-config")

if USE_CONFIG_FILE and not Path(CONFIG_PATH).exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, env=env)